In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
coeffs = [-1.0,-1.0]
observable = [qml.PauliZ(0),qml.PauliZ(1)]
H0 = qml.Hamiltonian(coeffs,observable)

In [3]:
device = qml.device('default.qubit',wires=2)
def ansatz(theta):
    qml.RX(theta[0],wires=0)
    qml.RY(theta[1],wires=1)
    qml.CNOT(wires=[0,1])
    qml.RY(theta[2],wires=0)
    qml.RX(theta[3],wires=1)

In [4]:
coeffs_p = [-1.0,1.0,-1.0]
observable_p = [qml.PauliZ(0)@qml.PauliZ(1),qml.PauliX(0),qml.PauliX(1)]
def make_H(s):
    coeff = [(1-s)*c for c in coeffs]+[s*c for c in coeffs_p]
    return qml.Hamiltonian(coeff,observable+observable_p)

In [5]:
def cost_function(s):
    h_s = make_H(s)
    @qml.qnode(device)
    def cost(theta):
        ansatz(theta)
        return qml.expval(h_s)
    return cost

In [6]:
s_steps = 50
s_value = np.linspace(0.0,1.0,s_steps)
theta = np.random.uniform(0.0,np.pi,4,requires_grad = True)
for i,s in enumerate(s_value):
    costs = cost_function(s)
    opt = qml.GradientDescentOptimizer(stepsize=0.2)
    for _ in range(150):
        theta,c = opt.step_and_cost(costs,theta)
    if i%10==0:
        print(f'current energy is {costs(theta)}')
print(f'final parameter is {theta}')
print()
print(f'gorund state energy of new hamitonian is {cost_function(1.0)(theta)}')

current energy is -2.0
current energy is -1.8372267299362042
current energy is -1.7543955331759993
current energy is -1.7485046645380244
current energy is -1.8246219075636476
final parameter is [-9.8813129e-324  1.4092479e+000 -1.4092479e+000  9.8813129e-324]

gorund state energy of new hamitonian is -1.9998304641621192
